# 05 - SCD Type 2 Validation

This notebook validates the SCD Type 2 implementation on `dim_restaurant`.

**SCD2 merge logic lives in Notebook 04** (part of the regular Gold pipeline).

**How to test:**
1. Run the pipeline (01 → 03 → 04) via Databricks Jobs — initial load
2. Modify source data (raw CSV in Volumes OR Bronze table)
3. Re-run the pipeline — SCD2 detects the change
4. Run this notebook to validate results

**SCD2 Details:**
- **Table**: `dim_restaurant`
- **Business Key**: `restaurant_name` + `source_city` + `license_number`
- **Tracked Attributes**: `facility_type`, `risk_category`, `aka_name`
- **SCD2 Columns**: `effective_start_date`, `effective_end_date`, `is_current`
- **Method**: Delta Lake MERGE (2-step: expire old, insert new)

In [0]:
%run ./00_setup_config

# 00 - Setup & Configuration
This notebook sets up the database, defines paths, and installs required libraries for the Food Inspection project.

All file paths are parameterized using widgets — no hardcoded values.

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Widget Parameters

Volume Path      : /Volumes/workspace/food_inspection/raw_data
Raw Chicago Path : /Volumes/workspace/food_inspection/raw_data/Food_Inspections_20260411.csv
Raw Dallas Path  : /Volumes/workspace/food_inspection/raw_data/Restaurant_and_Food_Establishment_Inspections_(October_2016_to_January_2024)_20260411.csv
Database Name    : food_inspection


## Create Database

Database 'food_inspection' is ready.


## Verify Raw Data Files Exist

Raw data files in Volume:
  Food_Inspections_20260411.csv (330.70 MB)
  Restaurant_and_Food_Establishment_Inspections_(October_2016_to_January_2024)_20260411.csv (192.79 MB)


Configuration ready. All path variables are available via %run.


In [0]:
spark.sql(f"USE {DATABASE_NAME}")

DataFrame[]

## 1. Overall dim_restaurant State
Shows count of current vs expired rows. After initial load, all rows are `is_current = True`.
After a pipeline re-run with changed data, some rows will be `is_current = False` (expired).

In [0]:
display(spark.sql(f"""
    SELECT is_current, COUNT(*) AS row_count
    FROM {DATABASE_NAME}.dim_restaurant
    GROUP BY is_current
    ORDER BY is_current DESC
"""))

total = spark.sql(f"SELECT COUNT(*) AS cnt FROM {DATABASE_NAME}.dim_restaurant").collect()[0]["cnt"]
current = spark.sql(f"SELECT COUNT(*) AS cnt FROM {DATABASE_NAME}.dim_restaurant WHERE is_current = True").collect()[0]["cnt"]
expired = total - current
print(f"Total rows: {total} | Current: {current} | Expired (historical): {expired}")

is_current,row_count
true,45548
false,2


Total rows: 45550 | Current: 45548 | Expired (historical): 2


## 2. Restaurants with SCD2 History
These are restaurants where a tracked attribute changed between pipeline runs.
Each should have at least 2 rows: one expired (old values) and one current (new values).

In [0]:
# Business key = restaurant_name + source_city + license_number
# Restaurants with >1 row per business key have SCD2 history
scd2_history = spark.sql(f"""
    SELECT restaurant_name, source_city, license_number,
           facility_type, risk_category, aka_name,
           effective_start_date, effective_end_date, is_current
    FROM {DATABASE_NAME}.dim_restaurant
    WHERE (restaurant_name, source_city, COALESCE(license_number, '')) IN (
        SELECT restaurant_name, source_city, COALESCE(license_number, '')
        FROM {DATABASE_NAME}.dim_restaurant
        GROUP BY restaurant_name, source_city, COALESCE(license_number, '')
        HAVING COUNT(*) > 1
    )
    ORDER BY restaurant_name, source_city, license_number, effective_start_date
""")

if scd2_history.count() > 0:
    print(f"Found {scd2_history.count()} rows with SCD2 history:")
    display(scd2_history)
else:
    print("No SCD2 history yet — run the pipeline with changed raw data to trigger SCD2.")

Found 4 rows with SCD2 history:


restaurant_name,source_city,license_number,facility_type,risk_category,aka_name,effective_start_date,effective_end_date,is_current
(K) NEW RESTAURANT,Chicago,1991508,SCD2_TEST_BAKERY,Risk 1 (High),(K) NEW RESTAURANT,2026-04-12,9999-12-31,true
(K) NEW RESTAURANT,Chicago,1991508,Restaurant,Risk 1 (High),(K) NEW RESTAURANT,2026-04-12,2026-04-12,false
1 2 3 EXPRESS,Chicago,1885022,Restaurant,Risk 3 (Low),SCD2_TEST_AKA_NAME,2026-04-12,9999-12-31,true
1 2 3 EXPRESS,Chicago,1885022,Restaurant,Risk 1 (High),1 2 3 EXPRESS,2026-04-12,2026-04-12,false


## 3. SCD2 Change Detail
Shows what specifically changed for restaurants with history.

In [0]:
scd2_detail = spark.sql(f"""
    WITH history AS (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY restaurant_name, source_city, COALESCE(license_number, '')
                ORDER BY effective_start_date DESC
            ) AS rn
        FROM {DATABASE_NAME}.dim_restaurant
        WHERE (restaurant_name, source_city, COALESCE(license_number, '')) IN (
            SELECT restaurant_name, source_city, COALESCE(license_number, '')
            FROM {DATABASE_NAME}.dim_restaurant
            GROUP BY restaurant_name, source_city, COALESCE(license_number, '')
            HAVING COUNT(*) > 1
        )
    )
    SELECT
        h_new.restaurant_name,
        h_new.source_city,
        h_new.license_number,
        h_old.facility_type AS old_facility_type,
        h_new.facility_type AS new_facility_type,
        h_old.risk_category AS old_risk_category,
        h_new.risk_category AS new_risk_category,
        h_old.aka_name AS old_aka_name,
        h_new.aka_name AS new_aka_name,
        h_old.effective_start_date AS old_start,
        h_old.effective_end_date AS old_end,
        h_new.effective_start_date AS new_start,
        h_new.effective_end_date AS new_end
    FROM history h_new
    JOIN history h_old
        ON h_new.restaurant_name = h_old.restaurant_name
        AND h_new.source_city = h_old.source_city
        AND COALESCE(h_new.license_number, '') = COALESCE(h_old.license_number, '')
        AND h_new.rn = 1 AND h_old.rn = 2
    ORDER BY h_new.restaurant_name
""")

if scd2_detail.count() > 0:
    display(scd2_detail)
else:
    print("No SCD2 changes detected yet — will populate after pipeline re-run with changed data.")

restaurant_name,source_city,license_number,old_facility_type,new_facility_type,old_risk_category,new_risk_category,old_aka_name,new_aka_name,old_start,old_end,new_start,new_end
(K) NEW RESTAURANT,Chicago,1991508,Restaurant,SCD2_TEST_BAKERY,Risk 1 (High),Risk 1 (High),(K) NEW RESTAURANT,(K) NEW RESTAURANT,2026-04-12,2026-04-12,2026-04-12,9999-12-31
1 2 3 EXPRESS,Chicago,1885022,Restaurant,Restaurant,Risk 1 (High),Risk 3 (Low),1 2 3 EXPRESS,SCD2_TEST_AKA_NAME,2026-04-12,2026-04-12,2026-04-12,9999-12-31


## 4. Validate SCD2 Integrity
Checks:
- Every restaurant has exactly 1 current row
- Expired rows have a valid end date (not 9999-12-31)
- Current rows have end date = 9999-12-31

In [0]:
# Check 1: Every business key (restaurant_name + source_city + license_number) should have exactly 1 current row
multi_current = spark.sql(f"""
    SELECT restaurant_name, source_city, license_number, COUNT(*) AS current_count
    FROM {DATABASE_NAME}.dim_restaurant
    WHERE is_current = True
    GROUP BY restaurant_name, source_city, license_number
    HAVING COUNT(*) > 1
""")

if multi_current.count() == 0:
    print("✓ CHECK PASSED: Every restaurant has exactly 1 current row")
else:
    print(f"✗ CHECK FAILED: {multi_current.count()} restaurants have multiple current rows")
    display(multi_current)

✓ CHECK PASSED: Every restaurant has exactly 1 current row


In [0]:
# Check 2: All expired rows should have end_date != 9999-12-31
bad_expired = spark.sql(f"""
    SELECT * FROM {DATABASE_NAME}.dim_restaurant
    WHERE is_current = False AND effective_end_date = '9999-12-31'
""")

if bad_expired.count() == 0:
    print("✓ CHECK PASSED: All expired rows have a valid end date")
else:
    print(f"✗ CHECK FAILED: {bad_expired.count()} expired rows still have end_date = 9999-12-31")

✓ CHECK PASSED: All expired rows have a valid end date


In [0]:
# Check 3: All current rows should have end_date = 9999-12-31
bad_current = spark.sql(f"""
    SELECT * FROM {DATABASE_NAME}.dim_restaurant
    WHERE is_current = True AND effective_end_date != '9999-12-31'
""")

if bad_current.count() == 0:
    print("✓ CHECK PASSED: All current rows have end_date = 9999-12-31")
else:
    print(f"✗ CHECK FAILED: {bad_current.count()} current rows have wrong end date")

✓ CHECK PASSED: All current rows have end_date = 9999-12-31


## 5. Delta Table History
Shows the version history of the dim_restaurant Delta table — each MERGE operation creates a new version.

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {DATABASE_NAME}.dim_restaurant"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-04-12T09:57:51.000Z,75299763588048,ravi.ara@northeastern.edu,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])","List(436327109340686, Food_Inspection_ETL_Pipeline, 451806265403617, 116590895477712, 75299763588048, manual)",null,a351f769-5dc4-41e3-b866-93e96fb21410,0412-084627-ki89yv90-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numOutputRows -> 6, numOutputBytes -> 4749)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
1,2026-04-12T09:57:45.000Z,75299763588048,ravi.ara@northeastern.edu,MERGE,"Map(predicate -> [""(((restaurant_name#101712 = restaurant_name#101510) AND (source_city#101717 = source_city#101502)) AND ((coalesce(license_number#101714, ) = coalesce(license_number#101512, )) AND is_current#101721))""], clusterBy -> [], matchedPredicates -> [{""predicate"":""((NOT (coalesce(facility_type#101715, ) = coalesce(facility_type#101513, )) OR NOT (coalesce(risk_category#101716, ) = coalesce(risk_category#101514, ))) OR NOT (coalesce(aka_name#101713, ) = coalesce(aka_name#101511, )))"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])","List(436327109340686, Food_Inspection_ETL_Pipeline, 451806265403617, 116590895477712, 75299763588048, manual)",null,18597459-d3d5-45ce-b160-f2d8e64959cb,0412-084627-ki89yv90-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 4409, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 4414, materializeSourceTimeMs -> 1023, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1163, numTargetRowsUpdated -> 2, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 45453, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2178)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
0,2026-04-12T09:47:18.000Z,75299763588048,ravi.ara@northeastern.edu,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)","List(436327109340686, Food_Inspection_ETL_Pipeline, 667978122421606, 890684100977969, 75299763588048, manual)",null,93f6c1c6-6e2a-4a89-8a31-211d5b273ff8,0412-084627-ki89yv90-v2n,null,WriteSerializable,true,"Map(numFiles -> 2, numOutputRows -> 45544, numOutputBytes -> 978640)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13


## 6. SCD2 Implementation Documentation

### Architecture:
- SCD2 merge logic is built into **Notebook 04** (Gold load pipeline)
- On first run: `dim_restaurant` is created with initial load (all `is_current = True`)
- On subsequent runs: Delta MERGE detects changes and applies SCD2

### How it works:
1. **Staging data** is built from Silver tables, deduplicated per business key using window functions (latest inspection values)
2. **MERGE Step 1**: Compare staging vs current dim rows — if tracked attributes changed, expire old row (`is_current = False`, `effective_end_date = today`)
3. **MERGE Step 2**: Insert new current row for changed/new restaurants (`is_current = True`, `effective_start_date = today`, `effective_end_date = 9999-12-31`)

### Testing Approach:
- **Test via Raw CSV**: Modify the source CSV in Volumes → Run full pipeline (01 → 03 → 04) → SCD2 detects change
- **Test via Bronze**: Modify Bronze Delta table directly → Run pipeline (03 → 04) → SCD2 detects change
- **Validation**: Run this notebook (05) to verify expired/current rows and integrity checks
- **Pipeline**: Set up in Databricks Jobs (01 → 03 → 04), validated via SQL Query Editor